In [ ]:
# %% [markdown]
# # Acoustic Drone Detection - Results Analysis
# **Author:** [Your Name]  
# **Date:** 2026-01-18  
# **Institution:** Istanbul Technical University

# %% [markdown]
# ## 1. Setup

# %%
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '..')

# %%
# Load checkpoints
experiments = {}
exp_dir = Path('../experiments')

for exp_path in exp_dir.iterdir():
    if exp_path.is_dir() and (exp_path / 'checkpoints' / 'best_model.pt').exists():
        ckpt = torch.load(exp_path / 'checkpoints' / 'best_model.pt', weights_only=False)
        experiments[exp_path.name] = ckpt
        print(f"Loaded: {exp_path.name}")

# %% [markdown]
# ## 2. Experiment Summary

# %%
print("=" * 60)
print("EXPERIMENT RESULTS SUMMARY")
print("=" * 60)

for name, ckpt in experiments.items():
    print(f"\n📊 {name}")
    print(f"   Best Epoch: {ckpt.get('epoch', 'N/A')}")
    print(f"   Val Loss:   {ckpt.get('best_val_loss', 'N/A'):.4f}" if ckpt.get('best_val_loss') else "   Val Loss: N/A")

# %% [markdown]
# ## 3. Dataset Distribution

# %%
# Dataset statistics (from training output)
datasets = {
    'Al-Emadi': {'Drone': 1332, 'No Drone': 10372},
    'DADS': {'Drone': 0, 'No Drone': 0},  # TODO: Update with real numbers
    'DroneThesis': {'Drone': 0, 'No Drone': 0},  # TODO: Update
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart for Al-Emadi
colors = ['#ff6b6b', '#4ecdc4']
axes[0].pie(
    [datasets['Al-Emadi']['Drone'], datasets['Al-Emadi']['No Drone']], 
    labels=['Drone', 'No Drone'],
    autopct='%1.1f%%',
    colors=colors,
    explode=(0.05, 0)
)
axes[0].set_title('Al-Emadi Dataset Distribution')

# Bar chart comparison
ax = axes[1]
x = np.arange(len(datasets))
width = 0.35
drone_counts = [d['Drone'] for d in datasets.values()]
no_drone_counts = [d['No Drone'] for d in datasets.values()]

ax.bar(x - width/2, drone_counts, width, label='Drone', color='#ff6b6b')
ax.bar(x + width/2, no_drone_counts, width, label='No Drone', color='#4ecdc4')
ax.set_xticks(x)
ax.set_xticklabels(datasets.keys())
ax.set_ylabel('Number of Samples')
ax.set_title('Dataset Comparison')
ax.legend()

plt.tight_layout()
plt.savefig('../docs/figures/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 4. Model Comparison Table

# %%
from IPython.display import Markdown

results_table = """
| Experiment | Model | Dataset | Best Epoch | Val Loss | Val Acc | F1 Score |
|------------|-------|---------|------------|----------|---------|----------|
| baseline_v1 | Custom CNN | Al-Emadi | 23 | 0.0650 | ~93% | TBD |
| combined_v1 | EfficientNet-B0 | Combined | 24 | 0.0359 | 99.11% | 0.9575 |
"""

Markdown(results_table)

# %% [markdown]
# ## 5. Spectrogram Examples

# %%
import librosa
import librosa.display

# Find sample audio files
alemadi_path = Path('../data/external/alemadi/Binary_Drone_Audio')
drone_files = list((alemadi_path / 'yes_drone').glob('*.wav'))[:1]
noise_files = list((alemadi_path / 'unknown').glob('*.wav'))[:1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, (files, title) in enumerate([(drone_files, 'Drone'), (noise_files, 'No Drone')]):
    if files:
        y, sr = librosa.load(files[0], sr=16000, duration=3.0)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        S_dB = librosa.power_to_db(S, ref=np.max)
        
        librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', ax=axes[idx])
        axes[idx].set_title(f'{title} - Mel Spectrogram')

plt.tight_layout()
plt.savefig('../docs/figures/spectrogram_examples.png', dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 6. Key Findings
# 
# 1. **EfficientNet-B0 outperforms Custom CNN** on combined dataset
# 2. **99.11% validation accuracy** achieved with pretrained model
# 3. **Dataset is imbalanced** - 88% no-drone vs 12% drone in Al-Emadi
# 4. **Next steps:** Cross-dataset validation to verify generalization